# VertiRTA-X8-240 — Colab G4 Headless Blender 5.2 LTS Cycles OptiX 4K `.vrta` Cinema

100% ToS-clean Layer-2 offline rendering pipeline targeting **Google Colab G4 (NVIDIA RTX PRO 6000 Blackwell, 96 GB VRAM)**.
- **Driver-Gated Blender LTS:** Installs **Blender 5.2 LTS** (`driver >= 575`) or **Blender 4.5 LTS** fallback (`535..574`); rejects 4.2/4.3 lacking Blackwell `sm_120` OptiX.
- **ToS-Clean Corridor Data:** Raw AWS Copernicus GLO-30 (`N40_00_E029_00`, `-dstnodata -9999`) + raw AWS `sentinel-cogs` L2A (`35TPF`) warped into Kartal local `tmerc` (NEVER `EPSG:32635`, EOX, FABDEM, Bing, Esri, or Google).
- **Resumable 150-Frame Slices:** Runs 3-frame `< 18 GB` VRAM `--cycles-print-stats` preflight, renders 12x 150-frame slices (`1..1800`) syncing to Drive, and muxes synced RGBA HUD via `hevc_nvenc -preset p5`.

In [ ]:
import subprocess
from viz.blender.blender_env_check import build_colab_install_commands, probe_host_nvidia_driver_major, select_blender_release

driver_major = probe_host_nvidia_driver_major()
release = select_blender_release(driver_major)
print(f"Detected NVIDIA driver {driver_major} -> Blender {release['version']} LTS (min_driver={release['min_driver']})")
for cmd in build_colab_install_commands(driver_major):
    print("$ ", cmd)
    subprocess.run(cmd, shell=True, check=True)


In [ ]:
from pathlib import Path
from viz.geo.airspace_polygons import export_airspace_geojson
from viz.geo.build_istanbul_corridor import build_corridor_pipeline_plan

plan = build_corridor_pipeline_plan("/content/work/geo_data")
export_airspace_geojson("/content/work/geo_data/istanbul_ltfj_ltbx_ctr.geojson")
print("Verified Kartal local tmerc CRS:", plan["local_tmerc_crs"])
print("GLO-30 tile:", plan["tiles"]["copernicus_glo30"], "| Sentinel-2 tile:", plan["tiles"]["sentinel2_mgrs"])


In [ ]:
# Build master 15x15 km Istanbul coastal .blend using procedural PBR shaders
# (ShaderNodeMix RGBA indices [6]/[7]/[2], Blender >=5 use_nodes guard, non-shadow LTFJ/LTBX curtains)
!blender -b --python-expr "import bpy; from viz.blender.build_istanbul_scene import build_turkish_rooftop_concrete_slz_shader, build_terracotta_roof_shader, build_marmara_ocean_shader, build_nishita_istanbul_sky, build_ctr_curtain_object; build_turkish_rooftop_concrete_slz_shader(bpy); build_terracotta_roof_shader(bpy); build_marmara_ocean_shader(bpy); build_nishita_istanbul_sky(bpy); build_ctr_curtain_object(bpy, 'ltfj'); build_ctr_curtain_object(bpy, 'ltbx'); bpy.ops.wm.save_as_mainfile(filepath='/content/work/vertirta_istanbul_master.blend')"


In [ ]:
import subprocess
from viz.blender.render_slices import assert_vram_within_budget, build_preflight_3frame_cmd, parse_cycles_peak_vram_gb

pf_cmd = build_preflight_3frame_cmd(
    "/content/work/vertirta_istanbul_master.blend",
    "viz/blender/apply_vrta_and_render_setup.py",
    "/content/preflight/pf_####",
    "/content/logs/kartal_m3_fault.vrta",
)
assert "OPTIX" in pf_cmd, "Cycles OPTIX device flag missing"
proc = subprocess.run(pf_cmd, capture_output=True, text=True, check=True)
peak_gb = parse_cycles_peak_vram_gb(proc.stdout + "\n" + proc.stderr)
assert_vram_within_budget(peak_gb, max_gb=18.0)
print(f"Preflight PASS: Peak VRAM = {peak_gb:.2f} GB (< 18.0 GB budget)")


In [ ]:
import subprocess
from pathlib import Path
from viz.blender.render_slices import build_blender_slice_cmd, build_slice_rsync_cmd, compute_frame_slices

out_dir = Path("/content/renders_4k")
drive_dir = Path("/content/drive/MyDrive/VertiRTA_4K_Frames")
completed = {int(p.stem.split("_")[-1]) for p in out_dir.glob("frame_*.png")} if out_dir.exists() else set()

for start_f, end_f in compute_frame_slices(total_frames=1800, slice_size=150, completed_frames=completed):
    cmd = build_blender_slice_cmd(
        "/content/work/vertirta_istanbul_master.blend",
        "viz/blender/apply_vrta_and_render_setup.py",
        str(out_dir / "frame_####"),
        "/content/logs/kartal_m3_fault.vrta",
        start_f,
        end_f,
    )
    subprocess.run(cmd, check=True)
    subprocess.run(build_slice_rsync_cmd(out_dir, drive_dir), check=True)


In [ ]:
import subprocess
from viz.blender.mux_hud_ffmpeg import build_ffmpeg_nvenc_cmd

ff_cmd = build_ffmpeg_nvenc_cmd(
    "/content/renders_4k/frame_%04d.png",
    "/content/hud_4k/hud_%04d.png",
    "/content/out/VertiRTA_X8_240_Istanbul_4K30_FlightReplay.mp4",
    fps=30,
    preset="p5",
)
print("Muxing 4K30 cinema:", " ".join(ff_cmd))
subprocess.run(ff_cmd, check=True)
